# Copula Project


## 1. Data Loading

The data has been processed in `data.py` module so we can directly load it from the CSV file.

In [47]:
import pandas as pd
data_file = "../data/sp500_log_returns.csv"

data = pd.read_csv(data_file, index_col=0, parse_dates=True)

data = data.fillna(0)



## 2. Generate simulated return rates

We fit copula using rolling windows of 10 years and refit every 1 year as the model takes time to fit.

In [48]:
# The code is in `main_copula.py` module in which we combined marginal fitting, copula fitting, and simulation.
from copula_model.main_copula import main_copula

sim_days = 253
n_paths = 1000
train_days = 252 * 10
refit_days = 253
all_simulated_returns, model_info, all_independent_returns = main_copula(data, n_paths=n_paths, sim_days=sim_days, 
                                                                         random_state=42, train_days=train_days, refit_freq=refit_days)

starting distribution fitting...
starting distribution fitting...starting distribution fitting...

starting distribution fitting...
starting distribution fitting...
starting distribution fitting...
Time elapsed: 19.514872074127197 seconds
starting copula fitting...
Time elapsed: 19.59998321533203 seconds
starting copula fitting...
Time elapsed: 19.692352056503296 seconds
starting copula fitting...
Time elapsed: 19.790016889572144 seconds
starting copula fitting...
Time elapsed: 19.845102071762085 seconds
starting copula fitting...
Time elapsed: 20.076768159866333 seconds
starting copula fitting...
Correlation matrix computed in 92.01558208465576 seconds
Gaussian log-likelihood computed in 0.055541038513183594 seconds
Correlation matrix computed in 92.38616800308228 seconds
Correlation matrix computed in 92.59832191467285 seconds
Gaussian log-likelihood computed in 0.056765079498291016 seconds
Gaussian log-likelihood computed in 0.05703926086425781 seconds


/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood


Correlation matrix computed in 92.9799268245697 seconds
Gaussian log-likelihood computed in 0.0560450553894043 seconds
Correlation matrix computed in 93.26377701759338 seconds
Gaussian log-likelihood computed in 0.05513811111450195 seconds
Correlation matrix computed in 93.02393794059753 seconds


/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood


Gaussian log-likelihood computed in 0.05491495132446289 seconds
t copula fitting computed in 18.79796290397644 seconds
t copula log-likelihood computed in 1.12510085105896 seconds
Selected copula: t
time elapsed: 113.00198173522949 seconds
starting simulation...
t copula fitting computed in 21.17241382598877 seconds
t copula log-likelihood computed in 1.006788969039917 seconds
Selected copula: t
time elapsed: 115.49818325042725 seconds
starting simulation...
t copula fitting computed in 27.69685196876526 seconds
t copula log-likelihood computed in 1.005599021911621 seconds
Selected copula: t
time elapsed: 121.73848795890808 seconds
starting simulation...
t copula fitting computed in 29.71744990348816 seconds
t copula log-likelihood computed in 1.032174825668335 seconds
Selected copula: t
time elapsed: 123.19262313842773 seconds
starting simulation...
t copula fitting computed in 34.54737687110901 seconds
t copula log-likelihood computed in 1.0337412357330322 seconds
Selected copula: t


In [49]:
import pickle

with open("../results/simulated_returns.pkl", "wb") as f:
    pickle.dump({
        "all_simulated_returns": all_simulated_returns,
        "model_info": model_info,
        "all_independent_returns": all_independent_returns
    }, f)

## 3. Backtesting VaR

We perform backtesting on the simulated return rates to evaluate the accuracy of the VaR estimates. Every 10 days, we calculate the $VaR_{99\%, 10 days}$ and $VaR_{95\%, 10 days}$ based on the simulated return rates and compare them with the actual returns to count the number of exceptions.

Also, we simulated return rates assuming independence among assets as a benchmark for comparison.


In [50]:
import pickle
with open("../results/simulated_returns.pkl", "rb") as f:
    results = pickle.load(f)
all_simulated_returns = results["all_simulated_returns"]
model_info = results["model_info"]
all_independent_returns = results["all_independent_returns"]


In [60]:
# Analyze the simulated returns
import numpy as np
for key in model_info.keys():
    # print(f"fitting date: {key}, copula: {model_info[key]['copula_name']}, nu: {model_info[key].get('nu', 'N/A')}")
    simulated_matrix = all_independent_returns[key]
    simulated_matrix_copula = all_simulated_returns[key]

    portfolio_returns = np.sum(simulated_matrix, axis=2) / simulated_matrix.shape[2]
    portfolio_returns_copula = np.sum(simulated_matrix_copula, axis=2) / simulated_matrix_copula.shape[2]

    low_quantile = 0.1
    high_quantile = 0.9
    independent_var_low = np.quantile(portfolio_returns, low_quantile)
    independent_var_high = np.quantile(portfolio_returns, high_quantile)
    copula_var_low = np.quantile(portfolio_returns_copula, low_quantile)
    copula_var_high = np.quantile(portfolio_returns_copula, high_quantile)

    print(f"fitting date: {key}, \n independent 1% quantile: {independent_var_low:.5f}, copula 1% quantile: {copula_var_low:.5f}, \n independent 99% quantile: {independent_var_high:.5f}, copula 99% quantile: {copula_var_high:.5f}")

fitting date: 2020-01-08 00:00:00, 
 independent 1% quantile: -0.00037, copula 1% quantile: -0.01014, 
 independent 99% quantile: 0.00180, copula 99% quantile: 0.01154
fitting date: 2021-01-08 00:00:00, 
 independent 1% quantile: -0.00065, copula 1% quantile: -0.01148, 
 independent 99% quantile: 0.00219, copula 99% quantile: 0.01308
fitting date: 2022-01-10 00:00:00, 
 independent 1% quantile: -0.00051, copula 1% quantile: -0.01067, 
 independent 99% quantile: 0.00214, copula 99% quantile: 0.01234
fitting date: 2023-01-12 00:00:00, 
 independent 1% quantile: -0.00061, copula 1% quantile: -0.01175, 
 independent 99% quantile: 0.00219, copula 99% quantile: 0.01335
fitting date: 2024-01-17 00:00:00, 
 independent 1% quantile: -0.00067, copula 1% quantile: -0.01213, 
 independent 99% quantile: 0.00214, copula 99% quantile: 0.01363
fitting date: 2025-01-21 00:00:00, 
 independent 1% quantile: -0.00063, copula 1% quantile: -0.01224, 
 independent 99% quantile: 0.00218, copula 99% quantile: 

In [ ]:
# Analyze the model info
for key in model_info.keys():
    print(f"fitting date: {key}, copula: {model_info[key]['copula_name']}, nu: {model_info[key].get('nu', 'N/A')}")


fitting date: 2020-01-08 00:00:00, copula: t, nu: 9.81936352241863
fitting date: 2021-01-08 00:00:00, copula: t, nu: 8.064782590961087
fitting date: 2022-01-10 00:00:00, copula: t, nu: 7.958202257490866
fitting date: 2023-01-12 00:00:00, copula: t, nu: 8.587323507169307
fitting date: 2024-01-17 00:00:00, copula: t, nu: 9.724872044082682
fitting date: 2025-01-21 00:00:00, copula: t, nu: 9.529783506131793


In [54]:
# Analyze the model info
for key in model_info.keys():
    corr_matrix = model_info[key]['corr_matrix']
    print(f"fitting date: {key}, correlation matrix:\n{corr_matrix}\n")

fitting date: 2020-01-08 00:00:00, correlation matrix:
[[1.         0.25135828 0.22668861 ... 0.2755526  0.36510618 0.18096865]
 [0.25135828 1.         0.19294253 ... 0.22807553 0.32770435 0.11335533]
 [0.22668861 0.19294253 1.         ... 0.21777834 0.31088217 0.45800343]
 ...
 [0.2755526  0.22807553 0.21777834 ... 1.         0.3645142  0.15124268]
 [0.36510618 0.32770435 0.31088217 ... 0.3645142  1.         0.28515786]
 [0.18096865 0.11335533 0.45800343 ... 0.15124268 0.28515786 1.        ]]

fitting date: 2021-01-08 00:00:00, correlation matrix:
[[1.         0.26263353 0.21670038 ... 0.26932472 0.33672786 0.16073783]
 [0.26263353 1.         0.23304742 ... 0.29520966 0.41469945 0.16328013]
 [0.21670038 0.23304742 1.         ... 0.2171963  0.30843546 0.47078713]
 ...
 [0.26932472 0.29520966 0.2171963  ... 1.         0.37279705 0.15226454]
 [0.33672786 0.41469945 0.30843546 ... 0.37279705 1.         0.26804008]
 [0.16073783 0.16328013 0.47078713 ... 0.15226454 0.26804008 1.        ]]



In [2]:
import pandas as pd
import numpy as np
import pickle

# paths
hist_path = "/Users/matthew/Documents/Rutgers/25 fall/RM/copula_model-main/data/sp500_log_returns.csv"
cop_path  = "/Users/matthew/Documents/Rutgers/25 fall/RM/copula_model-main/data/copula_simulated_returns.pkl"
ind_path  = "/Users/matthew/Documents/Rutgers/25 fall/RM/copula_model-main/data/independent_simulated_returns.pkl"

historical_returns = pd.read_csv(hist_path, index_col=0)
historical_returns.index = pd.to_datetime(historical_returns.index)
historical_returns = historical_returns.fillna(0)

print("Historical:", historical_returns.shape)

with open(cop_path, "rb") as f:
    copula_returns = pickle.load(f)

with open(ind_path, "rb") as f:
    indep_returns = pickle.load(f)

print("Number of refit dates:", len(copula_returns))
print("Example key:", list(copula_returns.keys())[:3])


Historical: (3982, 503)
Number of refit dates: 6
Example key: [Timestamp('2020-01-08 00:00:00'), Timestamp('2021-01-08 00:00:00'), Timestamp('2022-01-10 00:00:00')]


In [3]:
def portfolio_var(sim_matrix, alpha=0.01):
    """
    sim_matrix: (n_paths, horizon, n_assets)
    return: VaR series for each horizon day, shape (horizon,)
    """
    portfolio_paths = sim_matrix.mean(axis=2)
    var_series = np.quantile(portfolio_paths, alpha, axis=0)
    return var_series

hist = historical_returns
hist_idx = hist.index

records = []
sorted_dates = sorted(copula_returns.keys())

for refit_date in sorted_dates:
    sim_cop = copula_returns[refit_date]
    sim_ind = indep_returns[refit_date]

    n_paths, horizon, n_assets = sim_cop.shape

    var_cop_series = portfolio_var(sim_cop, alpha=0.01)
    var_ind_series = portfolio_var(sim_ind, alpha=0.01)

    if refit_date not in hist_idx:
        continue
    start_pos = hist_idx.get_loc(refit_date)

    for h in range(horizon):
        pos = start_pos + 1 + h
        if pos >= len(hist_idx):
            break

        realized_date = hist_idx[pos]
        realized_ret = hist.iloc[pos].mean()

        records.append({
            "VaR_date": refit_date,
            "Realized_date": realized_date,
            "Realized_return": realized_ret,
            "VaR_copula": var_cop_series[h],
            "VaR_indep": var_ind_series[h]
        })

var_df = pd.DataFrame(records).sort_values("Realized_date").reset_index(drop=True)
print("VaR table shape:", var_df.shape)
var_df.head()


VaR table shape: (1462, 5)


,VaR_date,Realized_date,Realized_return,VaR_copula,VaR_indep
0,2020-01-08,2020-01-09,0.005086,-0.029439,-0.001261
1,2020-01-08,2020-01-10,-0.002105,-0.027878,-0.001346
2,2020-01-08,2020-01-13,0.007479,-0.023175,-0.001509
3,2020-01-08,2020-01-14,0.000678,-0.024182,-0.001459
4,2020-01-08,2020-01-15,0.002215,-0.023109,-0.001395


In [4]:
def tl_color(n_exceptions, window=250):
    """
    Basel traffic-light thresholds for 250 days, 99% VaR.
    If window != 250, we still use the same cutoffs for illustration.
    """
    if n_exceptions <= 4:
        return "Green"
    elif n_exceptions <= 9:
        return "Yellow"
    else:
        return "Red"

var_df["hit_copula"] = (var_df["Realized_return"] < var_df["VaR_copula"]).astype(int)
var_df["hit_indep"]  = (var_df["Realized_return"] < var_df["VaR_indep"]).astype(int)

# last 250 days
window = 250
n_exc_copula_250 = int(var_df["hit_copula"].tail(window).sum())
n_exc_indep_250  = int(var_df["hit_indep"].tail(window).sum())

TL_copula_250 = tl_color(n_exc_copula_250, window)
TL_indep_250  = tl_color(n_exc_indep_250, window)

print("=== Traffic-Light Backtest (last 250 days) ===")
print(f"Copula model:      exceptions = {n_exc_copula_250:3d}, TL = {TL_copula_250}")
print(f"Independent model: exceptions = {n_exc_indep_250:3d}, TL = {TL_indep_250}")

# from 2020-01-08
start_date = pd.Timestamp("2020-01-08")
sub = var_df[var_df["VaR_date"] >= start_date].copy()

n_exc_copula_all = int(sub["hit_copula"].sum())
n_exc_indep_all  = int(sub["hit_indep"].sum())
window_all = len(sub)

TL_copula_all = tl_color(n_exc_copula_all, window_all)
TL_indep_all  = tl_color(n_exc_indep_all, window_all)

print(f"\n=== Traffic-Light Backtest (from {start_date.date()} to end, {window_all} days) ===")
print(f"Copula model:      exceptions = {n_exc_copula_all:3d}, TL = {TL_copula_all}")
print(f"Independent model: exceptions = {n_exc_indep_all:3d}, TL = {TL_indep_all}")


=== Traffic-Light Backtest (last 250 days) ===
Copula model:      exceptions =   3, TL = Green
Independent model: exceptions =  96, TL = Red

=== Traffic-Light Backtest (from 2020-01-08 to end, 1462 days) ===
Copula model:      exceptions =  33, TL = Red
Independent model: exceptions = 549, TL = Red


In [5]:
block_size = 250
start_date = pd.Timestamp("2020-01-08")
sub = var_df[var_df["VaR_date"] >= start_date].copy().reset_index(drop=True)

blocks = []
for start in range(0, len(sub), block_size):
    block = sub.iloc[start:start + block_size]
    if len(block) < block_size:
        break

    n_exc_cop = int(block["hit_copula"].sum())
    n_exc_ind = int(block["hit_indep"].sum())

    TL_cop = tl_color(n_exc_cop, block_size)
    TL_ind = tl_color(n_exc_ind, block_size)

    blocks.append({
        "block_id": len(blocks) + 1,
        "start_date": block["VaR_date"].iloc[0],
        "end_date": block["VaR_date"].iloc[-1],
        "n_days": len(block),
        "exc_copula": n_exc_cop,
        "TL_copula": TL_cop,
        "exc_indep": n_exc_ind,
        "TL_indep": TL_ind
    })

blocks_df = pd.DataFrame(blocks)
blocks_df

,block_id,start_date,end_date,n_days,exc_copula,TL_copula,exc_indep,TL_indep
0,1,2020-01-08,2020-01-08,250,20,Red,97,Red
1,2,2020-01-08,2021-01-08,250,0,Green,75,Red
2,3,2021-01-08,2022-01-10,250,10,Red,118,Red
3,4,2022-01-10,2023-01-12,250,0,Green,94,Red
4,5,2023-01-12,2024-01-17,250,0,Green,83,Red


In [6]:
start_date = pd.Timestamp("2020-01-08")
sub = var_df[var_df["VaR_date"] >= start_date].copy()

print("=== Sample info ===")
print("Number of observations:", len(sub))
print("Start date:", sub["VaR_date"].min())
print("End date:", sub["VaR_date"].max())
print()

for col in ["VaR_copula", "VaR_indep"]:
    s = sub[col]
    print(f"=== {col} ===")
    print("Mean:", s.mean())
    print("Median:", s.median())
    print("5% quantile:", s.quantile(0.05))
    print("95% quantile:", s.quantile(0.95))
    print("Min:", s.min())
    print("Max:", s.max())
    print()

diff = sub["VaR_copula"] - sub["VaR_indep"]

print("=== Difference: VaR_copula - VaR_indep ===")
print("Mean diff:", diff.mean())
print("Median diff:", diff.median())
print("Min diff:", diff.min())
print("Max diff:", diff.max())
print("Share of days where copula VaR is more negative:",
      (sub["VaR_copula"] < sub["VaR_indep"]).mean())
print()

print("=== Exceptions summary (99% VaR) ===")
print("Total exceptions (copula):", int(sub["hit_copula"].sum()))
print("Total exceptions (indep):", int(sub["hit_indep"].sum()))
print("Exception rate copula:", sub["hit_copula"].mean())
print("Exception rate indep:", sub["hit_indep"].mean())


=== Sample info ===
Number of observations: 1462
Start date: 2020-01-08 00:00:00
End date: 2025-01-21 00:00:00

=== VaR_copula ===
Mean: -0.03158425044159523
Median: -0.031467585679606405
5% quantile: -0.03983203735360827
95% quantile: -0.02402231878700913
Min: -0.05205624243067213
Max: -0.019786478568852697

=== VaR_indep ===
Mean: -0.0019531925956352597
Median: -0.002002524956301482
5% quantile: -0.0024888247784672713
95% quantile: -0.0012792945497494667
Min: -0.0033957415872687986
Max: -0.0009787449408841249

=== Difference: VaR_copula - VaR_indep ===
Mean diff: -0.029631057845959968
Median diff: -0.02951036626414151
Min diff: -0.049916642922453756
Max diff: -0.01843605285833498
Share of days where copula VaR is more negative: 1.0

=== Exceptions summary (99% VaR) ===
Total exceptions (copula): 33
Total exceptions (indep): 549
Exception rate copula: 0.022571819425444596
Exception rate indep: 0.3755129958960328
